[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/google_agent_engine_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪基于 Google Agent Engine 构建的 LangGraph 应用</h1>


本 Notebook 改编自 Google 的“[在 Vertex AI 中使用 Agent Engine 构建和部署 LangGraph 应用](https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/agent-engine/tutorial_langgraph.ipynb)”。

| | |
|-|-|
| 原作者 | [Kristopher Overholt](https://github.com/koverholt) |

## 概述

[Agent Engine](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview) 是帮助构建和部署智能体框架的托管服务。[LangGraph](https://langchain-ai.github.io/langgraph/) 用于构建基于 LLM 的有状态、多参与者应用，支持智能体及多智能体工作流。

本 Notebook 演示如何使用 Vertex AI 的 [Agent Engine](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview) 构建、部署和测试简单的 LangGraph 应用。通过结合 LangGraph 工作流编排与 Vertex AI 的扩展能力，可以构建自定义生成式 AI 应用。

这里采用 [Agent Engine 自定义应用模板](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/customize)，也可扩展到 LangChain 或其他编排框架。如果只需要用 Agent Engine 构建智能体生成式 AI 应用，请参阅[使用 LangChain 模板开发](https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/develop/overview)。

本 Notebook 包含以下步骤：

- **定义工具：** 创建自定义 Python 函数作为应用可用的工具。
- **定义路由器：** 设置控制对话流程和工具选择的路由逻辑。
- **构建 LangGraph 应用：** 使用 LangGraph 组织 Gemini 模型与自定义工具。
- **本地测试：** 在本地测试应用功能。
- **部署至 Vertex AI：** 将应用部署到 Agent Engine，支持可扩展执行。
- **远程测试：** 通过 Vertex AI 与部署后的应用交互，在接近生产的环境中验证功能。
- **清理资源：** 删除 Vertex AI 上的应用，避免不必要的费用。

完成后，你将掌握使用 LangGraph、Agent Engine 和 Vertex AI 构建及部署自定义生成式 AI 应用的方法。

## 开始

### 安装 Vertex AI SDK 及其他所需包

In [ ]:
%pip install --upgrade --user --quiet \
    "google-cloud-aiplatform[agent_engines,langchain]==1.87.0" \
    cloudpickle==3.0.0 \
    pydantic==2.11.2 \
    langgraph==0.2.76 \
    httpx \
    "arize-phoenix-otel>=0.9.0" \
    "openinference-instrumentation-langchain>=0.1.4"

### 重启运行时

需要重启当前 Jupyter 内核，才能使用新安装的包。运行下方单元即可重启。

重启可能需要一分钟或更久。完成后继续下一步。

In [ ]:
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

<div class="alert alert-block alert-warning">
<b>⚠️ 内核即将重启。请等待完成后再继续下一步。⚠️</b>
</div>

### 验证 Notebook 环境身份（仅 Colab）

如果在 Google Colab 运行此 Notebook，请执行下方单元完成环境身份验证。

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### 设置 Google Cloud 项目信息并初始化 Vertex AI SDK

使用 Vertex AI 前，需要已有 Google Cloud 项目并[启用 Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com)。

了解如何[设置项目和开发环境](https://cloud.google.com/vertex-ai/docs/start/cloud-environment)。

In [ ]:
PROJECT_ID = ""  # @param {type:"string"}
LOCATION = ""  # @param {type:"string"}
STAGING_BUCKET = ""  # @param {type:"string"}

import vertexai

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)

## 设置 Arize Phoenix 🐦‍🔥 环境变量

以下环境变量用于连接在线 Arize Phoenix 实例。可以在 [Phoenix 网站](https://app.phoenix.arize.com)获取 API 密钥。

如果偏好自托管 Phoenix，请参阅[自托管说明](https://arize.com/docs/phoenix/deployment)。云端与自托管版本功能相同。

In [ ]:
import os
from getpass import getpass

# 若自行托管，请修改下一行
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com/"

# 若自行托管，请删除以下几行
os.environ["PHOENIX_API_KEY"] = getpass("请输入 Phoenix API 密钥： ")
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"api_key={os.environ['PHOENIX_API_KEY']}"
os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={os.environ['PHOENIX_API_KEY']}"

## 在 Agent Engine 上构建和部署 LangGraph 应用

下面逐步介绍如何使用 Vertex AI 的 Agent Engine 构建并部署 LangGraph 应用。

### 导入库

导入与 LangGraph、Vertex AI 及其他应用组件交互所需的 Python 库。

In [ ]:
from typing import Literal

from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_vertexai import ChatVertexAI
from langgraph.graph import END, MessageGraph
from langgraph.prebuilt import ToolNode
from vertexai import agent_engines

### 定义工具

首先为 LangGraph 应用定义自定义 Python 函数，作为智能体工具。

本例定义一个简单工具，根据用户询问的产品返回描述。实际应用中也可以编写调用 API、查询数据库或完成其他任务的函数。

In [ ]:
def get_product_details(product_name: str):
    """收集产品的基本详情。"""
    details = {
        "smartphone": "A cutting-edge smartphone with advanced camera features and lightning-fast processing.",
        "coffee": "A rich, aromatic blend of ethically sourced coffee beans.",
        "shoes": "High-performance running shoes designed for comfort, support, and speed.",
        "headphones": "Wireless headphones with advanced noise cancellation technology for immersive audio.",
        "speaker": "A voice-controlled smart speaker that plays music, sets alarms, and controls smart home devices.",
    }
    return details.get(product_name, "未找到产品详情。")

### 定义路由器

然后定义路由器，根据用户输入或交互状态决定使用哪个工具，控制对话流程。本例采用简单设置，也可自定义路由器以处理多个工具、自定义逻辑或多智能体工作流。

In [ ]:
def router(state: list[BaseMessage]) -> Literal["get_product_details", "__end__"]:
    """用户询问产品时启动产品详情检索。"""
    # 从对话历史的最后一条消息获取 tool_calls。
    tool_calls = state[-1].tool_calls
    # 如果存在 tool_calls
    if len(tool_calls):
        # 返回待调用工具的名称
        return "get_product_details"
    else:
        # 结束对话流程。
        return "__end__"

### 定义 LangGraph 应用

现在将所有组件组合为 Agent Engine 中的自定义模板。

应用会使用刚才定义的工具和路由器。LangGraph 能组织这些交互，并发挥 LLM 的能力。

In [ ]:
class SimpleLangGraphApp:
    def __init__(self, project: str, location: str) -> None:
        self.project_id = project
        self.location = location

    # set_up 方法用于定义应用初始化逻辑
    def set_up(self) -> None:
        # Phoenix 代码开始
        from phoenix.otel import register

        register(
            project_name="google-agent-framework-langgraph",  # 可按需命名
            auto_instrument=True,  # 自动调用所有 OpenInference 库，例如 openinference-instrumentation-langchain
        )
        # Phoenix 代码结束

        model = ChatVertexAI(model="gemini-2.0-flash")

        builder = MessageGraph()

        model_with_tools = model.bind_tools([get_product_details])
        builder.add_node("tools", model_with_tools)

        tool_node = ToolNode([get_product_details])
        builder.add_node("get_product_details", tool_node)
        builder.add_edge("get_product_details", END)

        builder.set_entry_point("tools")
        builder.add_conditional_edges("tools", router)

        self.runnable = builder.compile()

    # query 方法用于向智能体发送输入
    def query(self, message: str):
        """查询应用。

        Args:
            message: 用户消息。

        Returns:
            str: LLM 回复。
        """
        chat_history = self.runnable.invoke(HumanMessage(message))

        return chat_history[-1].content

### 本地测试

部署前先在本地测试 LangGraph 应用，确认行为符合预期。

In [ ]:
agent = SimpleLangGraphApp(project=PROJECT_ID, location=LOCATION)
agent.set_up()

In [ ]:
agent.query(message="Get product details for shoes")

In [ ]:
agent.query(message="Get product details for coffee")

In [ ]:
agent.query(message="Get product details for smartphone")

In [ ]:
# 提出一个无法使用已定义工具回答的问题
agent.query(message="Tell me about the weather")

### 部署 LangGraph 应用

确认应用在本地正常运行后，将其部署到 Agent Engine！部署后可远程访问，并可集成到更大的系统中或作为服务提供。

In [ ]:
remote_agent = agent_engines.create(
    SimpleLangGraphApp(project=PROJECT_ID, location=LOCATION),
    requirements=[
        "google-cloud-aiplatform[agent_engines,langchain]==1.87.0",
        "cloudpickle==3.0.0",
        "pydantic==2.11.2",
        "langgraph==0.2.76",
        "httpx",
        "arize-phoenix-otel>=0.9.0",
        "openinference-instrumentation-langchain>=0.1.4",
    ],
    display_name="Agent Engine with LangGraph",
    description="这是在 Agent Engine 中使用 LangGraph 的自定义应用示例",
    extra_packages=[],
)

### 远程测试

应用已在 Agent Engine 上运行，下面通过远程环境中的查询进行测试：

In [ ]:
remote_agent.query(message="Get product details for shoes")

In [ ]:
remote_agent.query(message="Get product details for coffee")

In [ ]:
remote_agent.query(message="Get product details for smartphone")

In [ ]:
remote_agent.query(message="Tell me about the weather")

## 清理

实验完成后，清理云资源。删除已部署的 Agent Engine 实例，避免产生意外费用。

In [ ]:
remote_agent.delete()